# Module 4 — Supervised Contrastive Learning (Google Colab)

**Fake News Detection · XLM-RoBERTa + Supervised Contrastive Learning + Explainable AI**

## Runtime → Change runtime type → **T4 GPU**, then **Runtime → Run all**.

Module 3's embeddings are a **FROZEN, READ-ONLY** baseline (verified 18/18,
2026-08-09). This notebook reads them and never writes to `embeddings/` or
`outputs/module3/`.

| Step | What happens |
|---|---|
| 1-3 | Mount Drive, locate project, install dependencies |
| 4 | GPU check |
| 5 | **Pre-training gate** — halts if frozen inputs are wrong |
| 6 | Mechanics tests (22) — halts on any failure |
| 7 | **Training**: ablation → selection → final seeds → zero-shot |
| 8 | 20-check verification |
| 9 | Results summary |

Steps 5 and 6 are hard gates: if either fails, execution stops before any
GPU time is spent.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Locate the project

Edit `DRIVE_PROJECT_PATH` if the project lives elsewhere.

In [ ]:
import os, sys
from pathlib import Path

DRIVE_PROJECT_PATH = "/content/drive/MyDrive/Fake_News_Detection"
PROJECT_ROOT = Path(DRIVE_PROJECT_PATH)
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder not found at {PROJECT_ROOT}. "
        "Module 4 reads Module 3's frozen embeddings from Drive."
    )
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Working directory:", os.getcwd())


## 3. Dependencies

In [ ]:
!pip install -q transformers sentencepiece scikit-learn
print('Dependencies ready.')


## 4. GPU check

Module 4 trains on frozen 768-d vectors, so it is far cheaper than Module 3 —
minutes rather than half an hour. A GPU still helps.

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} ({p.total_memory/1e9:.1f} GB) | CUDA {torch.version.cuda}")
else:
    print("No GPU — training will still run correctly, just slower.")


## 5. Pre-training gate — frozen Module 3 inputs

Verifies counts (48,305 / 10,351 / 10,352 = 69,008), 768-d dimensions,
L2 normalisation, NaN/Inf, label and language domains, 5,408 Tamil rows,
record-ID uniqueness, zero cross-split leakage, and that `cls` and `mean`
share identical record IDs.

**Stops here if anything fails.** This is the gate that would have caught the
wrong-dataset substitution during Module 3.

In [ ]:
!python scripts/train_contrastive_module4.py --preflight-only


## 6. Mechanics tests

22 tests covering the SupCon loss (against hand-computed values), the
language-aware sampler, evaluation metrics, and the frozen-input gate's
own negative cases.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "tests/test_contrastive.py"], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print("--- stderr ---\n", r.stderr)
assert r.returncode == 0, "MECHANICS TESTS FAILED — not proceeding to training"
print("\nAll mechanics tests passed. Safe to train.")


## 7. Training

**Ablation (seed 42)** — 2 poolings × 2 sampling strategies, each with SupCon
and a capacity-matched CE-MLP, plus logistic-regression and majority-class
baselines. 

**Selection** on validation macro-F1 only.

**Final** — selected configuration re-run on seeds 42/43/44, reported as
mean ± std. **Test evaluated exactly once**, after selection.

**Zero-shot** — English-only training → Tamil test, reported separately.

Every experiment is recorded, including weak arms.

In [ ]:
!python scripts/train_contrastive_module4.py


## 8. Verification — 20 checks

Reads only what training produced. Never retrains.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/verify_module4_outputs.py"], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print("--- stderr ---\n", r.stderr)
print(f"exit code: {r.returncode}  (0 = verified)")


## 9. Results summary

In [ ]:
import json
from pathlib import Path
rep = PROJECT_ROOT / "outputs" / "module4"

v = rep / "final_verification_report.json"
if v.exists():
    p = json.loads(v.read_text(encoding="utf-8"))
    print("=" * 72); print(p["verdict"]); print("=" * 72)
    print(f"{p['n_passed']} passed / {p['n_failed']} failed / {p['n_unavailable']} unavailable")
    for r_ in p.get("failure_reasons", []): print("  -", r_)
    print()
    for c in p["checks"]: print(f"  [{c['status']:^11s}] {c['check']:2d}. {c['name']}")

m = rep / "evaluation_metrics.json"
if m.exists():
    d = json.loads(m.read_text(encoding="utf-8"))
    f = d.get("final", {})
    print("\n" + "=" * 72); print("SELECTED CONFIGURATION"); print("=" * 72)
    print(json.dumps(f.get("selected_configuration"), indent=2))
    print("\nTEST (mean ± std across seeds):")
    for k, val in f.get("test_across_seeds", {}).items():
        if isinstance(val, dict) and "mean" in val:
            print(f"  {k:18s} {val['mean']:.4f} ± {val['std']:.4f}")
    for lang in ("en", "ta"):
        st = f.get("test_per_language_across_seeds", {}).get(lang, {})
        mf = st.get("macro_f1")
        if mf: print(f"  test macro-F1 [{lang}] {mf['mean']:.4f} ± {mf['std']:.4f}")
    z = d.get("zero_shot")
    if z:
        ta = z["test_tamil_zero_shot"]["overall"]
        print(f"\nZERO-SHOT (en-only → Tamil test): macro-F1={ta['macro_f1']:.4f} acc={ta['accuracy']:.4f}")

print("\n" + "=" * 72); print("GENERATED FILES"); print("=" * 72)
for n in ["preflight_verification.json","training_history.json","evaluation_metrics.json",
          "ablation_results.json","module4_report.md","module4_config_used.yaml",
          "final_verification_report.json","final_verification_report.md"]:
    pth = rep / n
    print(f"  {'OK ' if pth.exists() else 'MISSING'}  outputs/module4/{n}"
          + (f"  ({pth.stat().st_size:,} B)" if pth.exists() else ""))
mdl = PROJECT_ROOT / "models" / "module4"
if mdl.exists():
    for w in sorted(mdl.glob("*.pt")): print(f"  OK   models/module4/{w.name}  ({w.stat().st_size:,} B)")


## Done

Module 4 is complete when step 8 reports **MODULE 4 VERIFIED SUCCESSFULLY**
(20/20).

Module 3 remains frozen: `embeddings/` and `outputs/module3/` were read only.